# TinyCeNN-LM — Colab Training, Health Check, Hugging Face Publish & Test

This notebook:
1. checks the Colab GPU;
2. clones `vtavakkoli/TinyCeNN-LM`;
3. installs the project;
4. logs in to Hugging Face **without hard-coding a token**;
5. trains the CeNN adapter on FineWeb;
6. inspects `training_report.json`;
7. publishes the best checkpoint to your Hugging Face account;
8. reloads the uploaded checkpoint from Hugging Face;
9. runs small generation tests.

> **Security:** store your Hugging Face token in **Colab → Secrets** as `HF_TOKEN`.
> Do not commit a real `hf_...` token into a public notebook or Git repository.

In [ ]:
# Check GPU
!nvidia-smi


In [ ]:
# Clone/update the repository
import pathlib

REPO_URL = "https://github.com/vtavakkoli/TinyCeNN-LM.git"
REPO_DIR = pathlib.Path("/content/TinyCeNN-LM")

if REPO_DIR.exists():
    %cd /content/TinyCeNN-LM
    !git pull --ff-only
else:
    %cd /content
    !git clone {REPO_URL}
    %cd /content/TinyCeNN-LM


In [ ]:
# Install TinyCeNN-LM and Hugging Face tooling
!python -m pip install -q --upgrade pip
!python -m pip install -q -e .
!python -m pip install -q --upgrade huggingface_hub


## Login into Hugging Face Hub

Create a **write** token in Hugging Face, then add it to Colab Secrets as `HF_TOKEN`.

The equivalent direct form is:

```python
from huggingface_hub import login
login("hf_REPLACE_WITH_YOUR_NEW_TOKEN")
```

For safety, this notebook reads the token from Colab Secrets instead of embedding it in source code.

In [ ]:
# Login into Hugging Face Hub
from huggingface_hub import login, HfApi

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    print("HF_TOKEN secret not found. Falling back to interactive login.")
    login()

api = HfApi()
hf_user = api.whoami()["name"]
print("Logged in as:", hf_user)


## Training configuration

The default is a short **1M-token smoke run**. Once it is healthy, change `MAX_TOKENS` to `10_000_000` or `50_000_000`.

For a T4/L4/A100 Colab GPU, these defaults are conservative.

In [ ]:
# Training settings
MAX_TOKENS = 1_000_000
CONTEXT_LENGTH = 256
BATCH_SIZE = 8
GRAD_ACCUM = 4
CENN_STEPS = 4
LEARNING_RATE = 0.002

EVAL_EVERY = 25
EVAL_BATCHES = 8
EVAL_BATCH_SIZE = 4

OUTPUT_DIR = "/content/TinyCeNN-LM/checkpoints/colab-tinycenn-base"


In [ ]:
# Train the CeNN adapter.
# Tiny-LLM and FineWeb are downloaded automatically and cached by Hugging Face.
import subprocess, shlex

cmd = [
    "python", "scripts/train_adapter.py",
    "--max-tokens", str(MAX_TOKENS),
    "--context-length", str(CONTEXT_LENGTH),
    "--batch-size", str(BATCH_SIZE),
    "--grad-accum", str(GRAD_ACCUM),
    "--steps", str(CENN_STEPS),
    "--learning-rate", str(LEARNING_RATE),
    "--eval-every", str(EVAL_EVERY),
    "--eval-batches", str(EVAL_BATCHES),
    "--eval-batch-size", str(EVAL_BATCH_SIZE),
    "--output-dir", OUTPUT_DIR,
]
print("Running:", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Inspect training health
import json, pathlib, math

report_path = pathlib.Path(OUTPUT_DIR) / "training_report.json"
with report_path.open() as f:
    report = json.load(f)

print(json.dumps(report, indent=2))

status = report.get("status")
if status == "diverged":
    raise RuntimeError("Training diverged. Do not publish this checkpoint.")

print("\nTraining status:", status)
print("Initial eval loss:", report.get("initial_eval_loss"))
print("Best eval loss:", report.get("best_eval_loss"))
print("Best perplexity:", report.get("best_perplexity"))
print("Relative improvement:", report.get("relative_best_improvement"))


## Prepare the best checkpoint for Hugging Face

The trainer saves a `-best` adapter directory when validation improves. If no separate best directory exists, the final adapter is used.

The notebook also writes tokenizer files, the training report, and a small model card into the upload folder.

In [ ]:
from pathlib import Path
import shutil, json
from transformers import AutoTokenizer

best_dir = Path(OUTPUT_DIR + "-best")
final_dir = Path(OUTPUT_DIR)

publish_dir = best_dir if best_dir.exists() else final_dir
print("Publishing from:", publish_dir)

# Ensure tokenizer files are present in the checkpoint repo.
tokenizer = AutoTokenizer.from_pretrained("arnir0/Tiny-LLM", use_fast=True)
tokenizer.save_pretrained(publish_dir)

# Copy the training report next to the best adapter.
if report_path.exists():
    shutil.copy2(report_path, publish_dir / "training_report.json")

model_card = f'''---
base_model: arnir0/Tiny-LLM
library_name: transformers
pipeline_tag: text-generation
tags:
- cenn
- tiny-llm
- language-modeling
- recurrent-neural-network
- parameter-efficient
- research
---

# TinyCeNN-LM Base

CeNN residual adapter trained on top of `arnir0/Tiny-LLM`.

- CeNN recurrent steps: {CENN_STEPS}
- Context length: {CONTEXT_LENGTH}
- Training token budget: {MAX_TOKENS:,}
- Health status: {report.get("status")}
- Initial eval loss: {report.get("initial_eval_loss")}
- Best eval loss: {report.get("best_eval_loss")}
- Best perplexity: {report.get("best_perplexity")}

This repository contains the TinyCeNN adapter weights/config plus tokenizer metadata.
Use the TinyCeNN-LM GitHub code to rebuild the base model + adapter.
'''
(publish_dir / "README.md").write_text(model_card, encoding="utf-8")

print("Upload folder contents:")
for p in sorted(publish_dir.iterdir()):
    print(" -", p.name)


In [ ]:
# Create a Hugging Face model repository and upload the best checkpoint.
# The repository name is derived automatically from the logged-in account.
HF_MODEL_NAME = "TinyCeNN-LM-Base"
HF_REPO_ID = f"{hf_user}/{HF_MODEL_NAME}"
HF_PRIVATE = False  # change to True if you want a private model repo

api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    private=HF_PRIVATE,
    exist_ok=True,
)

api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type="model",
    folder_path=str(publish_dir),
    commit_message=f"Upload TinyCeNN-LM checkpoint ({MAX_TOKENS:,} training tokens)",
)

print("Uploaded model:")
print(f"https://huggingface.co/{HF_REPO_ID}")


## Reload from Hugging Face and run small tests

This is important: the next cells do **not** use the local checkpoint path. They download the published adapter back from Hugging Face, reconstruct Tiny-LLM + CeNN, and generate text.

In [ ]:
# Download the checkpoint back from Hugging Face
from huggingface_hub import snapshot_download

downloaded_adapter = snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type="model",
)
print("Downloaded to:", downloaded_adapter)


In [ ]:
# Load the published model and run deterministic/small generation tests
import torch
from transformers import AutoTokenizer
from tinycenn_lm import build_from_adapter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = (
    torch.bfloat16
    if device.type == "cuda" and torch.cuda.is_bf16_supported()
    else (torch.float16 if device.type == "cuda" else torch.float32)
)

model = build_from_adapter(downloaded_adapter, device=device, dtype=dtype)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(downloaded_adapter)

prompts = [
    "The capital of Austria is",
    "Artificial intelligence can help",
    "A small language model",
    "In the future, efficient AI",
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    print("\nPROMPT:", prompt)
    print(tokenizer.decode(out[0], skip_special_tokens=True))


In [ ]:
# Basic numerical sanity test on the uploaded model
import math

test_text = "TinyCeNN-LM is a compact research language model."
batch = tokenizer(test_text, return_tensors="pt").to(device)

with torch.inference_mode():
    outputs = model(**batch, labels=batch["input_ids"], use_cache=False)

loss = float(outputs.loss.detach().cpu())
ppl = math.exp(min(loss, 20.0))

print(f"Sanity loss: {loss:.4f}")
print(f"Sanity perplexity: {ppl:.2f}")

assert math.isfinite(loss), "Non-finite loss after reload from Hugging Face."
print("Reload + inference sanity check: PASS")


## Next run

If the 1M-token run is healthy:
- change `MAX_TOKENS = 10_000_000`;
- rerun training through upload;
- then move to 50M tokens only if validation continues to improve.

The uploaded repository can also be tested locally with:

```bash
HF_MODEL_REPO=<your-hf-user>/TinyCeNN-LM-Base docker compose run --rm test-hf
```